**Ключевые моменты:**

* PySpark поддерживает чтение CSV-файлов с помощью запятых, табуляции, пробелов или любых других разделителей/разделителей.

* PySpark читает CSV-файлы параллельно, используя несколько узлов-исполнителей для ускорения загрузки данных.

* PySpark может автоматически определять схему CSV-файлов, что во многих случаях избавляет от необходимости определять схему вручную.

* Пользователи могут создавать собственные схемы для CSV-файлов, указывая типы данных и имена столбцов по мере необходимости.

* PySpark предлагает варианты обработки заголовков в CSV-файлах, позволяя пользователям пропускать заголовки или рассматривать их как строки данных.

* В программе предусмотрены надежные механизмы обработки ошибок при работе с неправильно сформированными или поврежденными CSV-файлами, обеспечивающие целостность данных.

In [1]:
import sys
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType,StructField,StringType,IntegerType,DoubleType

spark = (SparkSession.builder
         .appName("check-python")
         .master("local[*]")
         .config("spark.pyspark.python", sys.executable)
         .config("spark.pyspark.driver.python", sys.executable)
         .getOrCreate())

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/03/03 10:48:01 WARN Utils: Your hostname, MacBook--Azich.local, resolves to a loopback address: 127.0.0.1; using 172.20.10.2 instead (on interface en0)
26/03/03 10:48:01 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 10:48:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/03 10:48:02 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/03/03 10:48:02 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [5]:
df = spark.read.csv('zipcodes.csv')
df.show(truncate=False)

+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|_c0         |_c1    |_c2        |_c3                |_c4  |_c5           |_c6  |_c7    |_c8  |_c9  |_c10 |_c11       |_c12   |_c13                   |_c14                        |_c15         |_c16           |_c17               |_c18      |_c19         |
+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|RecordNumber|Zipcode|ZipCodeType|City               |State|LocationType  |Lat  |Long   |Xaxis|Yaxis|Zaxis|WorldRegion|Country|LocationText           |Location                    |Decommisioned|TaxReturnsFiled|EstimatedPopulation|To

В качестве альтернативы можно использовать format().load()

In [ ]:
# Using format().load()
df = spark.read.format("csv").load("zipcodes.csv")
df.printSchema()

root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)



In [14]:
df = spark.read.csv('zipcodes.csv', header=True)
df.show(truncate=False)

+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|RecordNumber|Zipcode|ZipCodeType|City               |State|LocationType  |Lat  |Long   |Xaxis|Yaxis|Zaxis|WorldRegion|Country|LocationText           |Location                    |Decommisioned|TaxReturnsFiled|EstimatedPopulation|TotalWages|Notes        |
+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|1           |704    |STANDARD   |PARC PARQUE        |PR   |NOT ACCEPTABLE|17.96|-66.22 |0.38 |-0.87|0.3  |NA         |US     |Parc Parque, PR        |NA-US-PR-PARC PARQUE        |FALSE        |NULL           |NULL               |NU

---
**delimiter**

Опция delimiter используется для указания разделителя столбцов в CSV-файле. По умолчанию это запятая (,), но с помощью этой опции можно установить любой символ, например, трубу (|), табуляцию (\t), пробел.

In [18]:
df3 = spark.read.options(delimiter=',') \
  .csv("zipcodes.csv", header=True)
df3.show(truncate=False)

+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|RecordNumber|Zipcode|ZipCodeType|City               |State|LocationType  |Lat  |Long   |Xaxis|Yaxis|Zaxis|WorldRegion|Country|LocationText           |Location                    |Decommisioned|TaxReturnsFiled|EstimatedPopulation|TotalWages|Notes        |
+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|1           |704    |STANDARD   |PARC PARQUE        |PR   |NOT ACCEPTABLE|17.96|-66.22 |0.38 |-0.87|0.3  |NA         |US     |Parc Parque, PR        |NA-US-PR-PARC PARQUE        |FALSE        |NULL           |NULL               |NU

---
**inferSchema**

По умолчанию для этой опции установлено значение False, при установке значения True она автоматически определяет типы столбцов на основе данных. Обратите внимание, что для вывода схемы требуется прочитать данные еще раз.

In [21]:
df4 = spark.read.options(inferSchema='True',delimiter=',') \
  .csv("zipcodes.csv", header=True)
df4.show(truncate=False)

+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|RecordNumber|Zipcode|ZipCodeType|City               |State|LocationType  |Lat  |Long   |Xaxis|Yaxis|Zaxis|WorldRegion|Country|LocationText           |Location                    |Decommisioned|TaxReturnsFiled|EstimatedPopulation|TotalWages|Notes        |
+------------+-------+-----------+-------------------+-----+--------------+-----+-------+-----+-----+-----+-----------+-------+-----------------------+----------------------------+-------------+---------------+-------------------+----------+-------------+
|1           |704    |STANDARD   |PARC PARQUE        |PR   |NOT ACCEPTABLE|17.96|-66.22 |0.38 |-0.87|0.3  |NA         |US     |Parc Parque, PR        |NA-US-PR-PARC PARQUE        |false        |NULL           |NULL               |NU

In [ ]:
# Using write options
df2.write.options(header='True', delimiter=',') \
 .csv("/tmp/spark_output/zipcodes")

---
**Режимы сохранения**

При записи PySpark DataFrame на диск можно задать различные режимы сохранения. Эти режимы сохранения определяют, как записывать файл на диск.

* overwrite - Перезаписать существующий файл, если он уже существует.

* append - Новые строки добавляются к существующим.

* ignore - При использовании этой опции операция записи игнорируется, если файл уже существует.

* error - Эта опция возвращает ошибку, если файл уже существует. Это опция по умолчанию.

In [ ]:
df.write.mode('overwrite').csv("/tmp/spark_output/zipcodes")

# You can also use this
df.write.format("csv").mode('overwrite').save("/tmp/spark_output/zipcodes")

In [22]:
spark.stop()